In [ ]:
# Imports

from minigrad.scalar import nn
from minigrad.scalar import ops
from minigrad.scalar import optim
from minigrad.scalar import Scalar

In [ ]:
# 1. Inputs

x1 = Scalar(1.0, name="x1")
x2 = Scalar(-1.0, name="x2")
y = Scalar(0.5, name="y")

print(x1, x2, y)

In [ ]:
# 2. Model definition

import random

random.seed(42)


def init_weight() -> float:
    return random.uniform(0.01, 0.1) * random.choice([-1, 1])


class Model(nn.Module):
    """Simple neural net."""

    def __init__(self) -> None:
        self._w11 = Scalar(init_weight(), name="w11")
        self._w12 = Scalar(init_weight(), name="w12")
        self._w2 = Scalar(init_weight(), name="w2")

        self._b1 = Scalar(0.0, name="b1")
        self._b2 = Scalar(0.0, name="b2")

        self._parameters = [self._w11, self._w12, self._w2, self._b1, self._b2]
        for p in self._parameters:
            p.requires_grad = True

        # Stores output of forward pass
        self.out: Scalar | None = None

    def forward(self, x1: Scalar, x2: Scalar) -> Scalar:
        z = x1 * self._w11 + x2 * self._w12 + self._b1
        h = ops.tanh(z)
        self._out = h * self._w2 + self._b2
        return self._out

In [ ]:
# 3. Optimization

learning_rate = 0.1
model = Model()
optimizer = optim.SGD(model.parameters, lr=learning_rate)
losses = []

In [ ]:
type(model.parameters)

In [ ]:
# 4. Training loop

from tqdm import tqdm

num_epochs = 1000

for epoch in tqdm(range(num_epochs)):
    # Forward pass
    y_pred = model(x1, x2)

    # Compute loss
    loss = 0.5 * (y_pred - y) ** 2
    losses.append(loss.data)

    # Early stopping condition
    if loss.data < 1e-15:
        print(f"Converged early at {epoch=} with loss {loss.data}")
        break

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
# 5. Plot losses
from matplotlib import pyplot as plt

plt.figure(figsize=(20, 10))
plt.plot(losses, label="Squared Loss")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss (Log Scale)")
plt.title("Perceptron Training Convergence")
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.legend()
plt.show()

In [ ]:
# 6. Make prediction

y_pred = model(x1, x2)
print(y_pred.data)